In [4]:
import pandas as pd
from pathlib import Path
import pyalex

data_dir = Path('../data')
cache_dir = data_dir / 'cache'
external_dir = data_dir / 'external'
processed_dir = data_dir / 'processed'

# Load the original DataFrame 

In [7]:
df = pd.read_parquet(external_dir / 'work_authorships.parquet')
df['work_openalex_id'].nunique()

334592

# Build the fact table (slim version)

In [41]:
df_main = df[['work_openalex_id', 'author_id', 'institution_id']].copy()
df_main

,work_openalex_id,author_id,institution_id
0,W1987491304,A5087092313,I4405264447
1,W1987491304,A5113715646,I4405264447
2,W1987491304,A5105839388,I4405264447
3,W1987491304,A5013200546,I134820265
4,W1987491304,A5013200546,I4210099652
...,...,...,...
4980142,W4303476441,A5047928635,I4210165077
4980143,W4303476441,A5047928635,I51713134
4980144,W4303476441,A5047928635,I83399316
4980145,W4303476441,A5016960321,I4210096275


#  Build the unique institution table (id + country)

In [42]:
df_inst = (
    df[['institution_id', 'institution_country']]
    .drop_duplicates(subset='institution_id')
    .reset_index(drop=True)
)
df_inst

,institution_id,institution_country
0,I4405264447,FR
1,I134820265,ES
2,I4210099652,ES
3,I4210165511,ES
4,I20581793,GB
...,...,...
30362,I4210162545,CN
30363,I4210159977,US
30364,I4210144718,PE
30365,I4210099012,CN


# Delete the original DataFrame to free memory 

In [43]:
del df

# Use pyalex to retrieve institution names

In [7]:
import time
from pyalex import Institutions, config
import pyalex
import pandas as pd

# Configure API key (required)
pyalex.config.api_key = "0EZyMj3Rml0QB4sngC7V7Z"

# Configure retries
config.max_retries = 3
config.retry_backoff_factor = 0.5
config.retry_http_codes = [429, 500, 503]

inst_ids = df_inst['institution_id'].tolist()
names_data = []

for i in range(0, len(inst_ids), 50):
    batch = inst_ids[i:i+50]
    try:
        results = Institutions().filter(openalex_id="|".join(batch)).get()
        for item in results:
            names_data.append({
                'institution_id': item['id'].split('/')[-1],
                'institution_name': item.get('display_name'),
            })
    except Exception as e:
        # If batch fails, log the IDs in this batch for later manual supplementation
        for bid in batch:
            names_data.append({'institution_id': bid, 'institution_name': None})
        print(f"Batch failed at {i}: {e}")
    
    print(f"fetched {i + len(batch)}/{len(inst_ids)}", end="\r")
    time.sleep(0.1)  # Maintain ~10 req/s

df_names = pd.DataFrame(names_data)
print(f"\nGot {len(df_names)} records")

fetched 30367/30367
Got 15192 records


In [44]:
# Here i manually remove duplicates because some problem occured during api calling and i dont know about apis
df_names = (
    df_names
    .drop_duplicates(subset='institution_id', keep='first')
    .reset_index(drop=True)
)
dup_mask = df_names.duplicated(subset='institution_id', keep=False)
print(f"Duplicate IDs in df_names: {df_names.loc[dup_mask, 'institution_id'].nunique()}")
df_names.loc[dup_mask].sort_values('institution_id').head(20)

Duplicate IDs in df_names: 0


,institution_id,institution_name


# Merge the names into the institution table 

In [45]:
df_inst = df_inst.merge(df_names, on='institution_id', how='left')
df_inst

,institution_id,institution_country,institution_name
0,I4405264447,FR,Université Pierre-et-Marie-Curie
1,I134820265,ES,Consejo Superior de Investigaciones Científicas
2,I4210099652,ES,NaN
3,I4210165511,ES,NaN
4,I20581793,GB,University of Kent
...,...,...,...
30362,I4210162545,CN,Children’s Hospital of Fudan University Xiamen...
30363,I4210159977,US,CentraCare Health System
30364,I4210144718,PE,Universidad de Ingeniería y Tecnología
30365,I4210099012,CN,Institute of Food Science and Technology


# Check coverage 

In [46]:
missing = df_inst[df_inst['institution_name'].isna()]
print(f"Missing names: {len(missing)}")
missing.head()

Missing names: 15178


,institution_id,institution_country,institution_name
2,I4210099652,ES,NaN
3,I4210165511,ES,NaN
5,I4210151560,ES,NaN
7,I55952717,ES,NaN
10,I4210147934,ES,NaN


# Write the two Parquet files 

In [47]:
df_main.to_parquet(external_dir / 'afiliations.parquet', index=False)
df_inst.to_parquet(external_dir / 'institutions.parquet', index=False)
print("Saved afiliations.parquet and institutions.parquet")

Saved afiliations.parquet and institutions.parquet


# Verify by reading back 

In [5]:
main_check = pd.read_parquet(external_dir / 'afiliations.parquet')
inst_check = pd.read_parquet(external_dir / 'institutions.parquet')

print("afiliations:", main_check.shape, list(main_check.columns))
print("institutions:", inst_check.shape, list(inst_check.columns))

afiliations: (4980147, 3) ['work_openalex_id', 'author_id', 'institution_id']
institutions: (30367, 3) ['institution_id', 'institution_country', 'institution_name']
